# 04 AI Text Generation

## Goal
This notebook generates a new text layer based on the literary text units processed in Notebook 02.

The notebook will:
- load the processed literary units
- use city groups, keywords, and emotional labels as inputs
- generate a new descriptive text layer for each unit
- save the generated text as a structured dataset

This notebook creates the AI-generated text branch of the project pipeline.

## Step 1 — Import libraries
Import the libraries needed for config loading, dataframe processing, and text generation.

In [1]:
from pathlib import Path
import json
import random
import pandas as pd

## Step 2 — Load the project config
Load the shared project paths created earlier.

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
CONFIG_PATH = PROJECT_ROOT / "src" / "project_config.json"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

PROCESSED_LITERARY_DIR = Path(config["processed_literary_dir"])

if "processed_ai_text_dir" in config:
    PROCESSED_AI_TEXT_DIR = Path(config["processed_ai_text_dir"])
else:
    PROCESSED_AI_TEXT_DIR = PROJECT_ROOT / "data" / "processed" / "ai_generated_text"
    PROCESSED_AI_TEXT_DIR.mkdir(parents=True, exist_ok=True)

print("Processed literary dir:", PROCESSED_LITERARY_DIR)
print("Processed AI text dir:", PROCESSED_AI_TEXT_DIR)

Processed literary dir: D:\Work\Workspace\Projects\Python\data-driven-surface\data\processed\literary_text
Processed AI text dir: D:\Work\Workspace\Projects\Python\data-driven-surface\data\processed\ai_generated_text


## Step 3 — Load the processed literary units
Read the CSV exported in Notebook 02.

In [3]:
processed_csv_path = PROCESSED_LITERARY_DIR / "invisible_cities_units_processed.csv"

if not processed_csv_path.exists():
    raise FileNotFoundError(
        f"Processed literary CSV not found: {processed_csv_path}\n"
        "Please run 02_literary_text_processing.ipynb first."
    )

unit_df = pd.read_csv(processed_csv_path)
unit_df.head()

,unit_id,city_group,text,clean_text,char_count,word_count,keywords,dominant_emotion,joy,sadness,fear,awe,desire
0,1,Cities & Memory 1,Leaving there and proceeding for three days to...,Leaving there and proceeding for three days to...,745,136,"['evening', 'leaving', 'proceeding', 'east', '...",joy,2,0,0,1,1
1,2,Cities & Memory 2,When a man rides a long time through wild regi...,When a man rides a long time through wild regi...,742,131,"['isidora', 'spiral', 'young', 'rides', 'wild'...",desire,0,0,0,0,1
2,3,Cities & Desire 1,There are two ways of describing the city of D...,There are two ways of describing the city of D...,1288,221,"['dorothea', 'quarters', 'morning', 'desert', ...",desire,0,0,0,0,1
3,4,Cities & Memory 3,"In vain, great-hearted Kublai, shall I attempt...","In vain, great-hearted Kublai, shall I attempt...",1684,306,"['past', 'zaira', 'tell', 'steps', 'streets', ...",neutral,0,0,0,0,0
4,5,Cities & Desire 2,"At the end of three days, moving southward, yo...","At the end of three days, moving southward, yo...",1368,248,"['anastasia', 'desire', 'sometimes', 'agate', ...",desire,1,0,0,0,4


## Step 4 — Parse keyword lists
Convert stored keyword strings back into Python lists if needed.

In [4]:
def parse_keyword_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    value = str(value).strip()

    if value.startswith("[") and value.endswith("]"):
        try:
            import ast
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass

    return []

unit_df["keywords"] = unit_df["keywords"].apply(parse_keyword_list)
unit_df[["unit_id", "city_group", "keywords"]].head()

,unit_id,city_group,keywords
0,1,Cities & Memory 1,"[evening, leaving, proceeding, east, reach, di..."
1,2,Cities & Memory 2,"[isidora, spiral, young, rides, wild, regions,..."
2,3,Cities & Desire 1,"[dorothea, quarters, morning, desert, caravan,..."
3,4,Cities & Memory 3,"[past, zaira, tell, steps, streets, telling, h..."
4,5,Cities & Desire 2,"[anastasia, desire, sometimes, agate, onyx, ch..."


## Step 5 — Define descriptive templates
These templates generate a new design-oriented text layer using:
- city group
- selected keywords
- emotional tone

In [5]:
emotion_openings = {
    "joy": [
        "A luminous urban scene emerges,",
        "A vivid and celebratory city unfolds,",
        "A radiant architectural vision appears,"
    ],
    "desire": [
        "A dreamlike urban landscape takes shape,",
        "A city of longing and projection unfolds,",
        "An imagined architectural world appears,"
    ],
    "fear": [
        "A tense and unstable city begins to emerge,",
        "An ominous urban environment takes form,",
        "A fragile and unsettling architecture appears,"
    ],
    "sadness": [
        "A fading and melancholic city appears,",
        "A quiet and eroding urban landscape unfolds,",
        "A mournful architectural scene takes shape,"
    ],
    "awe": [
        "A monumental and striking city rises,",
        "An overwhelming architectural vision appears,",
        "A vast and majestic urban form unfolds,"
    ],
    "neutral": [
        "An urban structure begins to form,",
        "A spatial composition takes shape,",
        "A city-like environment appears,"
    ]
}

emotion_closings = {
    "joy": [
        "The space suggests warmth, openness, and visual delight.",
        "Its atmosphere implies brightness and collective wonder.",
        "The overall impression is one of energy and vivid presence."
    ],
    "desire": [
        "The scene feels suspended between memory, fantasy, and anticipation.",
        "Its atmosphere suggests longing, projection, and imagined possibility.",
        "The spatial language evokes attraction, aspiration, and distant promise."
    ],
    "fear": [
        "The space conveys instability, uncertainty, and latent threat.",
        "Its atmosphere implies tension, unease, and structural fragility.",
        "The overall impression is one of risk, ambiguity, and pressure."
    ],
    "sadness": [
        "The atmosphere suggests absence, distance, and quiet deterioration.",
        "The space feels shaped by loss, memory, and gradual decline.",
        "Its visual language implies stillness, erosion, and emotional weight."
    ],
    "awe": [
        "The scene conveys scale, intensity, and architectural power.",
        "Its atmosphere suggests grandeur, monumentality, and reverence.",
        "The spatial impression is one of vastness and overwhelming presence."
    ],
    "neutral": [
        "The atmosphere remains observational and spatially descriptive.",
        "The result is a balanced and descriptive urban composition.",
        "The spatial language remains open, structured, and analytical."
    ]
}

## Step 6 — Define the text generation helper
Generate a new design-oriented descriptive paragraph from structured literary features.

In [6]:
def select_keywords(keywords, max_n=5):
    if not isinstance(keywords, list):
        return []
    return keywords[:max_n]

def generate_ai_text(city_group, keywords, dominant_emotion):
    keywords = select_keywords(keywords, max_n=5)
    keyword_phrase = ", ".join(keywords) if keywords else "urban forms and atmospheric details"

    opening_choices = emotion_openings.get(dominant_emotion, emotion_openings["neutral"])
    closing_choices = emotion_closings.get(dominant_emotion, emotion_closings["neutral"])

    opening = random.choice(opening_choices)
    closing = random.choice(closing_choices)

    body = (
        f"{opening} "
        f"It is shaped by elements such as {keyword_phrase}, "
        f"organised into a spatial condition associated with {city_group}. "
        f"The resulting environment can be interpreted as a speculative architectural scene "
        f"translated from literary description into visual form. "
        f"{closing}"
    )

    return body

## Step 7 — Test AI text generation on a small subset
Preview a few generated texts before processing the full dataset.

In [7]:
test_ai_df = unit_df.head(5).copy()

test_ai_df["ai_generated_text"] = test_ai_df.apply(
    lambda row: generate_ai_text(
        row["city_group"],
        row["keywords"],
        row["dominant_emotion"]
    ),
    axis=1
)

test_ai_df[[
    "unit_id",
    "city_group",
    "dominant_emotion",
    "keywords",
    "ai_generated_text"
]]

,unit_id,city_group,dominant_emotion,keywords,ai_generated_text
0,1,Cities & Memory 1,joy,"[evening, leaving, proceeding, east, reach, di...","A luminous urban scene emerges, It is shaped b..."
1,2,Cities & Memory 2,desire,"[isidora, spiral, young, rides, wild, regions,...","A dreamlike urban landscape takes shape, It is..."
2,3,Cities & Desire 1,desire,"[dorothea, quarters, morning, desert, caravan,...","A city of longing and projection unfolds, It i..."
3,4,Cities & Memory 3,neutral,"[past, zaira, tell, steps, streets, telling, h...","A city-like environment appears, It is shaped ..."
4,5,Cities & Desire 2,desire,"[anastasia, desire, sometimes, agate, onyx, ch...","A dreamlike urban landscape takes shape, It is..."


## Step 8 — Inspect the generated text samples
Print the first few generated texts in a more readable format.

In [8]:
for _, row in test_ai_df.iterrows():
    print(f"Unit {row['unit_id']} | {row['city_group']}")
    print(f"Emotion: {row['dominant_emotion']}")
    print(f"Keywords: {row['keywords']}")
    print("Generated text:")
    print(row["ai_generated_text"])
    print("-" * 100)

Unit 1 | Cities & Memory 1
Emotion: joy
Keywords: ['evening', 'leaving', 'proceeding', 'east', 'reach', 'diomira', 'sixty', 'silver', 'domes', 'bronze', 'statues', 'gods']
Generated text:
A luminous urban scene emerges, It is shaped by elements such as evening, leaving, proceeding, east, reach, organised into a spatial condition associated with Cities & Memory 1. The resulting environment can be interpreted as a speculative architectural scene translated from literary description into visual form. Its atmosphere implies brightness and collective wonder.
----------------------------------------------------------------------------------------------------
Unit 2 | Cities & Memory 2
Emotion: desire
Keywords: ['isidora', 'spiral', 'young', 'rides', 'wild', 'regions', 'feels', 'desire', 'finally', 'buildings', 'staircases', 'encrusted']
Generated text:
A dreamlike urban landscape takes shape, It is shaped by elements such as isidora, spiral, young, rides, wild, organised into a spatial condi

## Step 9 — Generate AI text for the full dataset
Once the test samples look reasonable, generate the full AI text layer.

In [9]:
unit_df["ai_generated_text"] = unit_df.apply(
    lambda row: generate_ai_text(
        row["city_group"],
        row["keywords"],
        row["dominant_emotion"]
    ),
    axis=1
)

unit_df.head()

,unit_id,city_group,text,clean_text,char_count,word_count,keywords,dominant_emotion,joy,sadness,fear,awe,desire,ai_generated_text
0,1,Cities & Memory 1,Leaving there and proceeding for three days to...,Leaving there and proceeding for three days to...,745,136,"[evening, leaving, proceeding, east, reach, di...",joy,2,0,0,1,1,"A vivid and celebratory city unfolds, It is sh..."
1,2,Cities & Memory 2,When a man rides a long time through wild regi...,When a man rides a long time through wild regi...,742,131,"[isidora, spiral, young, rides, wild, regions,...",desire,0,0,0,0,1,"An imagined architectural world appears, It is..."
2,3,Cities & Desire 1,There are two ways of describing the city of D...,There are two ways of describing the city of D...,1288,221,"[dorothea, quarters, morning, desert, caravan,...",desire,0,0,0,0,1,"A city of longing and projection unfolds, It i..."
3,4,Cities & Memory 3,"In vain, great-hearted Kublai, shall I attempt...","In vain, great-hearted Kublai, shall I attempt...",1684,306,"[past, zaira, tell, steps, streets, telling, h...",neutral,0,0,0,0,0,"A spatial composition takes shape, It is shape..."
4,5,Cities & Desire 2,"At the end of three days, moving southward, yo...","At the end of three days, moving southward, yo...",1368,248,"[anastasia, desire, sometimes, agate, onyx, ch...",desire,1,0,0,0,4,"A city of longing and projection unfolds, It i..."


## Step 10 — Save the AI-generated text dataset as CSV
Export the structured AI-generated text layer.

In [10]:
ai_text_csv_path = PROCESSED_AI_TEXT_DIR / "ai_generated_text_processed.csv"
unit_df.to_csv(ai_text_csv_path, index=False, encoding="utf-8-sig")

print("Saved AI-generated text CSV to:", ai_text_csv_path)

Saved AI-generated text CSV to: D:\Work\Workspace\Projects\Python\data-driven-surface\data\processed\ai_generated_text\ai_generated_text_processed.csv


## Step 11 — Save the AI-generated text dataset as JSON
Export a JSON version for later cross mapping and prompt construction.

In [11]:
ai_text_json_path = PROCESSED_AI_TEXT_DIR / "ai_generated_text_processed.json"

records = unit_df.to_dict(orient="records")

with open(ai_text_json_path, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=4, ensure_ascii=False)

print("Saved AI-generated text JSON to:", ai_text_json_path)

Saved AI-generated text JSON to: D:\Work\Workspace\Projects\Python\data-driven-surface\data\processed\ai_generated_text\ai_generated_text_processed.json


## Step 12 — Summary statistics
Preview the generated text layer and confirm the dataset size.

In [12]:
print("Number of AI-generated text units:", len(unit_df))
unit_df[[
    "unit_id",
    "city_group",
    "dominant_emotion",
    "ai_generated_text"
]].head(10)

Number of AI-generated text units: 45


,unit_id,city_group,dominant_emotion,ai_generated_text
0,1,Cities & Memory 1,joy,"A vivid and celebratory city unfolds, It is sh..."
1,2,Cities & Memory 2,desire,"An imagined architectural world appears, It is..."
2,3,Cities & Desire 1,desire,"A city of longing and projection unfolds, It i..."
3,4,Cities & Memory 3,neutral,"A spatial composition takes shape, It is shape..."
4,5,Cities & Desire 2,desire,"A city of longing and projection unfolds, It i..."
5,6,Cities & Signs 1,sadness,"A quiet and eroding urban landscape unfolds, I..."
6,7,Cities & Memory 4,desire,"An imagined architectural world appears, It is..."
7,8,Cities & Desire 3,neutral,"A city-like environment appears, It is shaped ..."
8,9,Cities & Signs 2,desire,"An imagined architectural world appears, It is..."
9,10,Thin Cities 1,desire,"An imagined architectural world appears, It is..."


## Step 13 — Summary
This notebook has:
- loaded the processed literary units
- generated a new AI text layer using structured literary features
- exported the AI-generated text dataset as CSV and JSON

Next notebook:
- `06_cross_mapping.ipynb`